# A09 - Fixed Cross-Validation Folds

Create one shared five-fold split for model selection inside the fixed random training set. The random test set and chronological test set are never used to construct these folds.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedKFold

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

data_path = project_root / 'data' / 'hotel_bookings_clean.csv'
train_path = project_root / 'data' / 'splits' / 'train_ids.csv'
test_path = project_root / 'data' / 'splits' / 'test_ids.csv'

df = pd.read_csv(data_path)
train_ids = pd.read_csv(train_path)['row_id']
test_ids = pd.read_csv(test_path)['row_id']

train_df = df[df['row_id'].isin(train_ids)].copy()
assert len(train_df) == 94851
assert set(train_df['row_id']) == set(train_ids)
assert set(train_ids).isdisjoint(set(test_ids))

print('Training records available for CV:', len(train_df))
print('Random test records excluded from CV:', len(test_ids))

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_folds = pd.DataFrame({
    'row_id': train_df['row_id'].to_numpy(),
    'cv_fold': 0
})

for fold_number, (_, validation_index) in enumerate(
    cv.split(train_df, train_df['is_canceled']),
    start=1
):
    cv_folds.loc[validation_index, 'cv_fold'] = fold_number

print(cv_folds['cv_fold'].value_counts().sort_index())

In [ ]:
fold_path = project_root / 'data' / 'splits' / 'cv_folds.csv'
fold_path.parent.mkdir(parents=True, exist_ok=True)
cv_folds.to_csv(fold_path, index=False)

saved_folds = pd.read_csv(fold_path)
folded = train_df[['row_id', 'is_canceled']].merge(
    saved_folds, on='row_id', validate='one_to_one'
)

assert len(saved_folds) == 94851
assert saved_folds['row_id'].is_unique
assert set(saved_folds['row_id']) == set(train_ids)
assert set(saved_folds['row_id']).isdisjoint(set(test_ids))
assert set(saved_folds['cv_fold']) == {1, 2, 3, 4, 5}
assert saved_folds['cv_fold'].notna().all()

fold_summary = folded.groupby('cv_fold').agg(
    records=('row_id', 'size'),
    cancellation_rate=('is_canceled', 'mean')
)
fold_summary['cancellation_rate'] = fold_summary['cancellation_rate'].round(6)

print('Saved to:', fold_path)
print(fold_summary)
print('All fixed-fold validation checks passed.')

## Use in model selection

C, D, and E should merge `cv_folds.csv` into the fixed random training data by `row_id`. The fold number indicates which records are the validation set in each round. Do not use either held-out test set for cross-validation or hyperparameter tuning.

In [ ]:
from sklearn.model_selection import PredefinedSplit

training_with_folds = train_df.merge(
    saved_folds, on='row_id', validate='one_to_one'
)
predefined_cv = PredefinedSplit(
    test_fold=training_with_folds['cv_fold'].to_numpy() - 1
)

assert predefined_cv.get_n_splits() == 5
print('PredefinedSplit is ready with', predefined_cv.get_n_splits(), 'folds.')